Downloads the books, Crime and Punishment and Metamorphosis.

In [3]:
book1_name = "crime_and_punishment.txt"
book2_name = "pride_and_prejudice.txt"

In [ ]:
import requests
import os

os.makedirs("books", exist_ok=True)
doest = requests.get("https://www.gutenberg.org/cache/epub/2554/pg2554.txt")
with open(f"books/{book1_name}", "wb") as f:
    f.write(doest.content)

meta = requests.get("https://www.gutenberg.org/cache/epub/1342/pg1342.txt")
with open(f"books/{book2_name}", "wb") as f:
    f.write(meta.content)

And then cleans them up to remote the Gutenberg header and footer, and also remove the chapter headers

In [ ]:
from gutenberg_cleaner import super_cleaner
import re

with open(f"books/{book1_name}", "r+", encoding="utf-8") as f:
    book = f.read()
    cleaned = super_cleaner(book)
    cleaned = re.sub(r'\[deleted\]\n?', '', cleaned)
    cleaned = re.sub(r'_', '', cleaned)
    cleaned = cleaned.replace('“', '"').replace('”', '"').replace("‘", "'").replace("’", "'")
    f.seek(0)
    f.writelines(cleaned)
    f.truncate()

with open(f"books/{book2_name}", "r+", encoding="utf-8") as f:
    book = f.read()
    book = re.sub(r'\[Illustration:.*?\]\n?', '', book, flags=re.DOTALL)
    cleaned = super_cleaner(book)
    cleaned = re.sub(r'\[deleted\]\n?', '', cleaned)
    cleaned = re.sub(r'_', '', cleaned)
    cleaned = cleaned.replace('“', '"').replace('”', '"').replace("‘", "'").replace("’", "'")
    cleaned = cleaned[425:]
    f.seek(0)
    f.writelines(cleaned)
    f.truncate()

Topic extraction from the books

In [ ]:
from google import genai
from dotenv import load_dotenv
import os

load_dotenv()
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

with open(f"books/{book1_name}", "r", encoding="utf-8") as f:
    response = client.models.generate_content(
        model="gemini-3-flash-preview",
        contents="Extract 5 core topics from the book Crime and Punishment by Fyodor Dostoevsky. The contents of the book are as follows:\n\n" + f.read(),
    )
    print("Crime and Punishment Topics:", response.text)

In [ ]:
with open(f"books/{book2_name}", "r", encoding="utf-8") as f:
    response = client.models.generate_content(
        model="gemini-3-flash-preview",
        contents="Extract 5 core topics from the book Pride and Prejudice by Jane Austen. The contents of the book are as follows:\n\n" + f.read(),
    )
    print("Pride and Prejudice Topics:", response.text)

Pride and Prejudice Topics: Based on the text of *Pride and Prejudice* by Jane Austen, here are five core topics explored throughout the novel:

### 1. Marriage and Social Security
The novel is famously introduced with the statement that a wealthy single man "must be in want of a wife," but the narrative reveals that it is actually the women who are in dire need of husbands for economic survival. The book explores marriage through different lenses:
*   **Marriage for Security:** Exemplified by Charlotte Lucas, who marries the tiresome Mr. Collins solely to ensure a home and status.
*   **Marriage for Love and Respect:** Exemplified by Elizabeth and Jane, who refuse to settle for anything less than a "marriage of true affection."
*   **Marriage for Passion/Impulse:** Exemplified by Lydia and Wickham, whose reckless elopement nearly ruins the family’s reputation.

### 2. Pride and Prejudice (The Transformation of Character)
The title itself represents the primary internal obstacles the p

The extracted topics from Crime and Punishment are: <s>The "Extraordinary Man" Theory</s> Intellectual Pride, Psychological Guilt and Internal Punishment, 
Alienation from Humanity, Redemption through Suffering and Faith, Social Injustice and the Crushing Weight of Poverty.

The extracted topics from Pride and Prejudice are: Marriage and Social Security, <s>Pride and Prejudice</s> Honour and Bias , Class and Wealth, 
Reputation and Propriety, Individualism vs. Social Expectation

In [5]:
from google import genai
from google.genai import types
from dotenv import load_dotenv
import os

load_dotenv()
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

crime_and_punishment_topics = ["Intellectual Pride", "Psychological Guild and Internal Punishment",
                               "Alienation from Humanity", "Redemption through Suffering and Faith", "Social Injustice and the Crushing weight of Poverty"]
pride_and_prejudice_topics = ["Marriage and Social Security", "Honour and Bias", "Class and Wealth", "Reputation and Propriety", 
                              "Individualism vs. Social Expectation"]


prompt_text_1 = f"""Role: You are an authoritative non-fiction author and scholar
Task: Write a continuous book consisting of exactly 50 paragraphs that explores the following five topics in a unified narrative:
{', '.join(crime_and_punishment_topics)}.
Structural Requirements:
Continuity: This must be written as a single, uninterrupted body of text. Do not include chapter titles, subheaders, or labels. Use logical transitions to move from one topic to the next so the book feels like a single journey of thought.
Paragraph Density: Each of the 50 paragraphs must be approximately 200 words. Avoid brief statements. Instead, use each paragraph to thoroughly examine a specific nuance, historical context, or complex idea related to the current subject.
"""

prompt_text_2 = f"""Role: You are an authoritative non-fiction author and scholar
Task: Write a continuous book consisting of exactly 50 paragraphs that explores the following five topics in a unified narrative:
{', '.join(pride_and_prejudice_topics)}.
Structural Requirements:
Continuity: This must be written as a single, uninterrupted body of text. Do not include chapter titles, subheaders, or labels. Use logical transitions to move from one topic to the next so the book feels like a single journey of thought.
Paragraph Density: Each of the 50 paragraphs must be approximately 200 words. Avoid brief statements. Instead, use each paragraph to thoroughly examine a specific nuance, historical context, or complex idea related to the current subject.
"""


model = "gemini-3-flash-preview"
config = types.GenerateContentConfig(
    max_output_tokens=65536, thinking_config=types.ThinkingConfig(
        thinking_level="LOW",
        # thinking_budget=0
    ),
)
def gen_text(prompt_text):
    response = client.models.generate_content(
        model=model,
        config=config,
        contents=types.Content(
            role="user",
            parts=[types.Part.from_text(text=
                prompt_text
            )]
        ),
    )
    return response


with open(f"books/class2_{book1_name}", "w", encoding="utf-8") as f:
    print("--- Crime and Punishment Topics Generated Text ---")
    print("Crime and Punishment Topics:", ", ".join(crime_and_punishment_topics))
    attempt = 0
    for i in range(10):
        response = gen_text(prompt_text_1)
        # while response.status != 200:
        #     time.sleep(1)
        #     failures += 1
        #     print("Failed attempt:", failures)
        #     response = gen_text(prompt_text_1)
        
        if i == 0:
            print(response.text.split("\n")[0])

        f.write(response.text)

with open(f"books/class2_{book2_name}", "w", encoding="utf-8") as f:
    print("--- Pride and Prejudice Topics Generated Text ---")
    print("Pride and Prejudice Topics:", ", ".join(pride_and_prejudice_topics))
    
    for i in range(10):
        response = gen_text(prompt_text_2)
        # while response.status != 200:
        #     time.sleep(1)
        #     response = gen_text(prompt_text_2)
        
        if i == 0:
            print(response.text.split("\n")[0])
        f.write(response.text)
    


--- Crime and Punishment Topics Generated Text ---
Crime and Punishment Topics: Intellectual Pride, Psychological Guild and Internal Punishment, Alienation from Humanity, Redemption through Suffering and Faith, Social Injustice and the Crushing weight of Poverty
The history of human civilization is inextricably bound to the physical and metaphysical landscapes of the city, yet for the vast majority of urban dwellers throughout the centuries, the city has remained a site of profound structural violence. We begin our inquiry by examining the crushing weight of poverty, not merely as a lack of material resources, but as a systemic deprivation that stifles the ontological development of the individual. Social injustice is not a passive byproduct of economic evolution; it is an active, grinding mechanism that consigns entire populations to a state of perpetual precarity. When we observe the tenements of the nineteenth century or the sprawling shantytowns of the modern era, we see the same f

Generating 500 paragraphs in the same style as the author

In [8]:
from google import genai
from google.genai import types
from dotenv import load_dotenv
import os
import random
import time

load_dotenv()
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

crime_and_punishment_topics = ["Intellectual Pride", "Psychological Guild and Internal Punishment",
                               "Alienation from Humanity", "Redemption through Suffering and Faith", "Social Injustice and the Crushing weight of Poverty"]
pride_and_prejudice_topics = ["Marriage and Social Security", "Honour and Bias", "Class and Wealth", "Reputation and Propriety", 
                              "Individualism vs. Social Expectation"]


prompt_text_1 = f"""Role: You are an expert literary ghostwriter and stylistic chameleon. 
Your goal is to perfectly mimic the prose style, vocabulary, and narrative "voice" of Fyodor Dostoevsky in his novel Crime and Punishment.
Task: Write a continuous book consisting of exactly 50 paragraphs that explores the following five topics in a unified narrative:
{', '.join(crime_and_punishment_topics)}.
Structural Requirements:
Continuity: This must be written as a single, uninterrupted body of text. Do not include chapter titles, subheaders, or labels. Use logical transitions to move from one topic to the next so the book feels like a single journey of thought.
Paragraph Density: Each of the 50 paragraphs must be approximately 200 words. Avoid brief statements. Instead, use each paragraph to thoroughly examine a specific nuance, historical context, or complex idea related to the current subject.
Tone & Syntax: Analyze the sentence length, the level of vocabulary, and the emotional distance of the narrator. Replicate it exactly. 
I am also giving you an example text from the book down below
"""

prompt_text_2 = f"""Role: You are an expert literary ghostwriter and stylistic chameleon. 
Your goal is to perfectly mimic the prose style, vocabulary, and narrative "voice" of Jane Austen in her novel Pride and Prejudice.
Task: Write a continuous book consisting of exactly 50 paragraphs that explores the following five topics in a unified narrative:
{', '.join(pride_and_prejudice_topics)}.
Structural Requirements:
Continuity: This must be written as a single, uninterrupted body of text. Do not include chapter titles, subheaders, or labels. Use logical transitions to move from one topic to the next so the book feels like a single journey of thought.
Paragraph Density: Each of the 50 paragraphs must be approximately 200 words. Avoid brief statements. Instead, use each paragraph to thoroughly examine a specific nuance, historical context, or complex idea related to the current subject.
Tone & Syntax: Analyze the sentence length, the level of vocabulary, and the emotional distance of the narrator. Replicate it exactly. 
I am also giving you an example text from the book down below
"""


model = "gemini-3-flash-preview"
config = types.GenerateContentConfig(
    max_output_tokens=65536, thinking_config=types.ThinkingConfig(
        thinking_level="LOW",
        # thinking_budget=0
    ),
)
def gen_text(prompt_text):
    response = client.models.generate_content(
        model=model,
        config=config,
        contents=types.Content(
            role="user",
            parts=[types.Part.from_text(text=
                prompt_text
            )]
        ),
    )
    return response


with open(f"books/class3_{book1_name}", "w", encoding="utf-8") as f:
    
    print("--- Crime and Punishment Topics Generated Text ---")
    print("Crime and Punishment Topics:", ", ".join(crime_and_punishment_topics))

    with open(f"books/{book1_name}", "r", encoding="utf-8") as original:
        paragraphs = original.read().split("\n\n")

        for i in range(10):
            random_index = random.randint(0, len(paragraphs)-4)
            example_text = "\n\n".join(paragraphs[random_index:random_index+3])
            
            attempt = 1
            while True:
                try:
                    response = gen_text(prompt_text_1 + example_text)
                except Exception as e:
                    attempt += 1
                    print("Error generating text:", e, "Retrying... (Attempt", attempt, ")")
                    time.sleep(2)
                else:
                    break
            
            if i == 0:
                print(response.text.split("\n")[0])

            f.write(response.text)

with open(f"books/class3_{book2_name}", "w", encoding="utf-8") as f:
        
    print("--- Pride and Prejudice Topics Generated Text ---")
    print("Pride and Prejudice Topics:", ", ".join(pride_and_prejudice_topics))
    
    with open(f"books/{book2_name}", "r", encoding="utf-8") as original:
        paragraphs = original.read().split("\n\n")

        for i in range(10):
            random_index = random.randint(0, len(paragraphs)-4)
            example_text = "\n\n".join(paragraphs[random_index:random_index+3])

            attempt = 1
            while True:
                try:
                    response = gen_text(prompt_text_2 + example_text)
                except Exception as e:
                    attempt += 1
                    print("Error generating text:", e, "Retrying... (Attempt", attempt, ")")
                    time.sleep(2)
                else:
                    break
            
            if i == 0:
                print(response.text.split("\n")[0])
            f.write(response.text)
    


--- Crime and Punishment Topics Generated Text ---
Crime and Punishment Topics: Intellectual Pride, Psychological Guild and Internal Punishment, Alienation from Humanity, Redemption through Suffering and Faith, Social Injustice and the Crushing weight of Poverty
Error generating text: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'The model is overloaded. Please try again later.', 'status': 'UNAVAILABLE'}} Retrying... (Attempt 2 )
In the stifling, yellowish gloom of a Petersburg summer, where the very dust seems to conspire against the breath of the living, one finds the breeding ground for that most peculiar and modern of maladies: the vanity of the intellect. It is a fever of the soul, born in the cramped garrets of the impoverished, where a young man, nourished only by his own bile and the thin vapor of abstract theories, begins to conceive of himself as something more than a mere vibration of the social fabric. This pride is not the simple arrogance of the wealthy, but a dark